**Task 1:**
Classification Using K-Nearest Neighbour Classifier.

**1- Libraries Imports:**

In [66]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score
import math
import warnings
import scipy.stats
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)

**2- Data Preprocessing:**
- Reading the data
- Splitting the two classes
- Removing extra data
- Split features and target

In [67]:
data = pd.read_csv("magic04.data", header= None)
print(data.head())

         0         1       2       3       4         5        6        7   \
0   28.7967   16.0021  2.6449  0.3918  0.1982   27.7004  22.0110  -8.2027   
1   31.6036   11.7235  2.5185  0.5303  0.3773   26.2722  23.8238  -9.9574   
2  162.0520  136.0310  4.0612  0.0374  0.0187  116.7410 -64.8580 -45.2160   
3   23.8172    9.5728  2.3385  0.6147  0.3922   27.2107  -6.4633  -7.1513   
4   75.1362   30.9205  3.1611  0.3168  0.1832   -5.5277  28.5525  21.8393   

        8         9  10  
0  40.0920   81.8828  g  
1   6.3609  205.2610  g  
2  76.9600  256.7880  g  
3  10.4490  116.7370  g  
4   4.6480  356.4620  g  


In [68]:
data.isnull().sum()

0     0
1     0
2     0
3     0
4     0
5     0
6     0
7     0
8     0
9     0
10    0
dtype: int64

In [69]:
gamma = data[data[10] == 'g']
hadron =  data[data[10] == 'h']
print(f"gammas = {len(gamma)}")
print(f"hadrons = {len(hadron)}")

gammas = 12332
hadrons = 6688


In [70]:
gamma_sampled = gamma.sample(n=len(hadron), random_state=42)

print(f"gammas = {len(gamma_sampled)}")
print(f"hadrons = {len(hadron)}")

balanced_data = pd.concat([gamma_sampled, hadron])

gammas = 6688
hadrons = 6688


In [71]:
features = balanced_data.drop(columns=[10])
target = balanced_data[10]

**3- Data Splitting:**
- Split data to 70% train, 15% validation, and 15% test.

In [72]:
features_train, features_temp, target_train, target_temp = train_test_split(
    features,
    target,
    test_size= 0.3,
    shuffle= True,
    random_state= 42
)

In [73]:
features_test, features_validation, target_test, target_validation = train_test_split(
    features_temp,
    target_temp,
    test_size= 0.5,
    shuffle= True,
    random_state= 42
)

In [74]:
print(f"Length of train samples = {len(features_train)}")
print(f"Length of test samples = {len(features_validation)}")
print(f"Length of validation samples = {len(features_test)}")

Length of train samples = 9363
Length of test samples = 2007
Length of validation samples = 2006


**4- Apply K-NN Classifier to the data**  
We will apply K-Nearest Neighbour in 2 ways (self-written functions & using libraries)
In each method we will:
- Apply k = sqrt(n) "Common Heuristic".
- Apply different k values to get the best results.

1 - Self-written

In [75]:
def eculidean_distance (x,y):
    distance = np.linalg.norm(x-y, axis=1)
    return distance

In [76]:
def KNN(new_data_point, X = features_train.values, Y = target_train.values, k= int(round(math.sqrt(len(features_train))))):
    distances = eculidean_distance(X, new_data_point)
    nearest_neighbor_ids = distances.argsort()[:k]
    nearest_neighbor_target = Y[nearest_neighbor_ids]
    prediction = scipy.stats.mode(nearest_neighbor_target)[0]
    return prediction

In [81]:
validation_predict_sqrt = np.array([KNN(x) for x in features_validation.values])
validation_accuracy_sqrt = accuracy_score(target_validation, validation_predict_sqrt)
print(f"Validation Accuracy - Self-written function KNN (k=sqrt(n)): {validation_accuracy_sqrt:.2f}")

Validation Accuracy - Self-written function KNN (k=sqrt(n)): 0.76


2 - Using library sklearn

In [82]:
k = int(round(math.sqrt(len(features_train))))
print (f"k = {k}")

sqrtk_model = KNeighborsClassifier(n_neighbors= k)
sqrtk_model.fit(features_train, target_train)

validation_predict_sqrtsklearn = sqrtk_model.predict(features_validation)
validation_accuracy_sqrtsklearn = accuracy_score(target_validation, validation_predict_sqrtsklearn)
print(f"Validation Accuracy - sklearn KNN (k=sqrt(n)): {validation_accuracy_sqrtsklearn:.2f}")

k = 97
Validation Accuracy - sklearn KNN (k=sqrt(n)): 0.76


Applying different K to get the best in both models:

In [83]:
k_values = range(1, int(round(math.sqrt(len(features_train)))))
accuracies_without_sklearn = []
accuracies_sklearn = []

for k in k_values:

    # 1 - sklearn
    model = KNeighborsClassifier(n_neighbors=k)
    model.fit(features_train, target_train)
    y_pred_sklearn = model.predict(features_validation)
    accuracies_sklearn.append(accuracy_score(target_validation, y_pred_sklearn))

    # 2 - self-written function
    y_pred_without_sklearn = np.array([KNN(x , k= k) for x in features_validation.values])
    accuracies_without_sklearn.append(accuracy_score(target_validation, y_pred_without_sklearn))


In [84]:
best_k_without_sklearn = k_values[accuracies_without_sklearn.index(max(accuracies_without_sklearn))]
best_k_sklearn = k_values[accuracies_sklearn.index(max(accuracies_sklearn))]

print(f"Best K - sklearn: {best_k_sklearn}, Accuracy: {max(accuracies_sklearn):.2f}")
print(f"Best K - without sklearn: {best_k_without_sklearn}, Accuracy: {max(accuracies_without_sklearn):.2f}")

Best K - sklearn: 48, Accuracy: 0.77
Best K - without sklearn: 60, Accuracy: 0.77
